# Resultado - Hospital Habilitado



In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import unicodedata
import re

import pandas as pd
import plotly.express as px
import plotly.io as pio

if "vscode" in pio.renderers:
    pio.renderers.default = "vscode"
else:
    pio.renderers.default = "notebook_connected"

try:
    import geobr
except ImportError as exc:
    raise ImportError("Pacote 'geobr' não encontrado. Instale com: pip install geobr") from exc

import json
from IPython.display import display

In [ ]:
def normalizar_texto(valor: str) -> str:
    if pd.isna(valor):
        return ""
    txt = str(valor).strip().lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("utf-8")
    txt = re.sub(r"\s+", " ", txt)
    return txt

candidatos_base = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
path_hospitais = None
path_municipios = None

for base in candidatos_base:
    ph = base / "Tratamento" / "Hospitais" / "tabela_hospitais_habilitados.csv"
    pm = base / "Tratamento" / "Municípios" / "tabela_municipios_regionais.csv"
    if ph.exists() and pm.exists():
        path_hospitais = ph
        path_municipios = pm
        break

if path_hospitais is None or path_municipios is None:
    raise FileNotFoundError(
        "Não foi possível localizar tabela_hospitais_habilitados.csv e tabela_municipios_regionais.csv"
    )

df_hosp = pd.read_csv(path_hospitais)
df_mun_reg = pd.read_csv(path_municipios)

muni_mg = geobr.read_municipality(code_muni="MG", year=2020, simplified=True).to_crs(4326)

In [ ]:
df_hosp.columns = [c.strip() for c in df_hosp.columns]
df_mun_reg.columns = [c.strip() for c in df_mun_reg.columns]

mapa_tipos = {
    "oncologia": "Oncologia",
    "reconstrucao mamaria": "Reconstrução Mamária",
    "tratamento sincronico": "Tratamento Sincrônico",
}

df_hosp["CODIGO_IBGE"] = pd.to_numeric(df_hosp["CÓDIGO"], errors="coerce").astype("Int64")
df_hosp["MUNICIPIO_NORM"] = df_hosp["MUNICÍPIOS"].map(normalizar_texto)
df_hosp["TIPO_CLASSIFICADO"] = df_hosp["TIPO"].map(normalizar_texto).map(mapa_tipos)
df_hosp = df_hosp[df_hosp["TIPO_CLASSIFICADO"].notna()].copy()

df_mun_reg["codigo_ibge"] = pd.to_numeric(df_mun_reg["codigo_ibge"], errors="coerce").astype("Int64")
df_mun_reg["municipio_normalizado"] = df_mun_reg["municipio_normalizado"].fillna("").map(normalizar_texto)
df_mun_reg["regional_original"] = df_mun_reg["regional_original"].fillna("Nenhum")

mapa_regional_por_nome = dict(
    df_mun_reg[["municipio_normalizado", "regional_original"]]
    .drop_duplicates(subset=["municipio_normalizado"])
    .values
)

base_hosp = df_hosp.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="CODIGO_IBGE",
    right_on="codigo_ibge",
    how="left",
)
base_hosp["regional_assoc"] = base_hosp["regional_original"]
mask_sem_regional = base_hosp["regional_assoc"].isna() | base_hosp["regional_assoc"].eq("")
base_hosp.loc[mask_sem_regional, "regional_assoc"] = base_hosp.loc[mask_sem_regional, "MUNICIPIO_NORM"].map(mapa_regional_por_nome)
base_hosp["regional_assoc"] = base_hosp["regional_assoc"].fillna("Nenhum")

ordem_legenda = [
    "Reconstrução Mamária",
    "Reconstrução Mamária e Oncologia",
    "Reconstrução Mamária e Tratamento Sincrônico",
    "Reconstrução Mamária, Tratamento Sincrônico, Oncologia",
    "Tratamento Sincrônico",
    "Oncologia",
    "Nenhum",
]

def combinar_tipos(series_tipos):
    tipos_set = set([t for t in series_tipos if pd.notna(t)])

    if tipos_set == {"Reconstrução Mamária"}:
        return "Reconstrução Mamária"
    if tipos_set == {"Reconstrução Mamária", "Oncologia"}:
        return "Reconstrução Mamária e Oncologia"
    if tipos_set == {"Reconstrução Mamária", "Tratamento Sincrônico"}:
        return "Reconstrução Mamária e Tratamento Sincrônico"
    if tipos_set == {"Reconstrução Mamária", "Tratamento Sincrônico", "Oncologia"}:
        return "Reconstrução Mamária, Tratamento Sincrônico, Oncologia"
    if tipos_set == {"Tratamento Sincrônico"}:
        return "Tratamento Sincrônico"
    if tipos_set == {"Oncologia"}:
        return "Oncologia"

    return "Nenhum"

combo_municipio = (
    base_hosp.groupby("CODIGO_IBGE", as_index=False)
    .agg(
        categoria_mapa=("TIPO_CLASSIFICADO", combinar_tipos),
        tipos_presentes=("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
        estabelecimentos_unicos=("ESTABELECIMENTO", "nunique"),
        registros=("ESTABELECIMENTO", "count"),
    )
)

gdf_municipios = muni_mg.merge(
    combo_municipio,
    left_on="code_muni",
    right_on="CODIGO_IBGE",
    how="left",
)

gdf_municipios["categoria_mapa"] = gdf_municipios["categoria_mapa"].fillna("Nenhum")
gdf_municipios["tipos_presentes"] = gdf_municipios["tipos_presentes"].fillna("Nenhum")
gdf_municipios["estabelecimentos_unicos"] = gdf_municipios["estabelecimentos_unicos"].fillna(0).astype(int)
gdf_municipios["registros"] = gdf_municipios["registros"].fillna(0).astype(int)
gdf_municipios["categoria_mapa"] = pd.Categorical(gdf_municipios["categoria_mapa"], categories=ordem_legenda, ordered=True)

gdf_municipios_plot = gdf_municipios.reset_index(drop=True).copy()
gdf_municipios_plot["id"] = gdf_municipios_plot.index.astype(str)

combo_regional = (
    base_hosp[base_hosp["regional_assoc"] != "Nenhum"]
    .groupby("regional_assoc", as_index=False)
    .agg(
        categoria_mapa=("TIPO_CLASSIFICADO", combinar_tipos),
        tipos_presentes=("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
        estabelecimentos_unicos=("ESTABELECIMENTO", "nunique"),
        registros=("ESTABELECIMENTO", "count"),
    )
)

gdf_mg_reg = muni_mg.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="code_muni",
    right_on="codigo_ibge",
    how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

gdf_regionais = (
    gdf_mg_reg[gdf_mg_reg["regional_original"] != "Nenhum"][["regional_original", "geometry"]]
    .dissolve(by="regional_original", as_index=False)
)

gdf_regionais = gdf_regionais.merge(
    combo_regional,
    left_on="regional_original",
    right_on="regional_assoc",
    how="left",
)

gdf_regionais["categoria_mapa"] = gdf_regionais["categoria_mapa"].fillna("Nenhum")
gdf_regionais["tipos_presentes"] = gdf_regionais["tipos_presentes"].fillna("Nenhum")
gdf_regionais["estabelecimentos_unicos"] = gdf_regionais["estabelecimentos_unicos"].fillna(0).astype(int)
gdf_regionais["registros"] = gdf_regionais["registros"].fillna(0).astype(int)
gdf_regionais["categoria_mapa"] = pd.Categorical(gdf_regionais["categoria_mapa"], categories=ordem_legenda, ordered=True)

gdf_regionais_plot = gdf_regionais.reset_index(drop=True).copy()
gdf_regionais_plot["id"] = gdf_regionais_plot.index.astype(str)

In [ ]:
# cores_tipos = {
#     "Reconstrução Mamária": "#74a8db",
#     "Reconstrução Mamária e Oncologia": "#dc7b7b",
#     "Reconstrução Mamária e Tratamento Sincrônico": "#8e1e1e",
#     "Reconstrução Mamária, Tratamento Sincrônico, Oncologia": "#d73f3f",
#     "Tratamento Sincrônico": "#3e78b3",
#     "Oncologia": "#30475e",
#     "Nenhum": "#d3edf9",
# }
cores_tipos = {
    "Reconstrução Mamária": "#74a8db",
    "Reconstrução Mamária e Oncologia": "#ff7f50",
    "Reconstrução Mamária e Tratamento Sincrônico": "#8e1e1e",
    "Reconstrução Mamária, Tratamento Sincrônico, Oncologia": "#9c2328",
    "Tratamento Sincrônico": "#3e78b3",
    "Oncologia": "#fec44f",
    "Nenhum": "#ffffd4",
}



def centro(gdf):
    bounds = gdf.total_bounds 
    return (bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2

# ── Mapa por município 
lat_c, lon_c = centro(gdf_municipios_plot)

fig_municipios = px.choropleth_map(
    gdf_municipios_plot,
    geojson=json.loads(gdf_municipios_plot.to_json()),  
    locations="id",
    color="categoria_mapa",
    color_discrete_map=cores_tipos,
    category_orders={"categoria_mapa": ordem_legenda},
    title="Distribuição de Hospitais Habilitados por Município em Minas Gerais",
    hover_name="name_muni",
    hover_data={
        "tipos_presentes": True,
        "estabelecimentos_unicos": True,
        "registros": True,
        "id": False,
    },
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},  
    zoom=5.5,                               
    opacity=0.95,
)

# update_geos removido — não tem efeito em choropleth_map
fig_municipios.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=760,
    legend_title_text="Combinação",
)
fig_municipios.write_html("Gráficos/hospitais_municipios.html", include_plotlyjs="cdn", auto_open=False)

# ── Mapa por regional 
lat_c, lon_c = centro(gdf_regionais_plot)

fig_regionais = px.choropleth_map(
    gdf_regionais_plot,
    geojson=json.loads(gdf_regionais_plot.to_json()),    
    locations="id",
    color="categoria_mapa",
    color_discrete_map=cores_tipos,
    category_orders={"categoria_mapa": ordem_legenda},
    title="Distribuição de Hospitais Habilitados por Regional em Minas Gerais",
    hover_name="regional_original",
    hover_data={
        "tipos_presentes": True,
        "estabelecimentos_unicos": True,
        "registros": True,
        "id": False,
    },
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},   
    zoom=5.5,
    opacity=0.95,
)

fig_regionais.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=760,
    legend_title_text="Combinação",
)
fig_regionais.write_html("Gráficos/hospitais_regionais.html", include_plotlyjs="cdn", auto_open=False)
print("Mapas salvos em Gráficos/hospitais_municipios.html e Gráficos/hospitais_regionais.html")

In [ ]:
# Tabelas: quantidade de hospitais (estabelecimentos únicos) por município e por regional

tabela_municipios_hospitais = (
    base_hosp.assign(
        municipio=base_hosp["MUNICÍPIOS"].fillna("").astype(str).str.strip(),
        estabelecimento=base_hosp["ESTABELECIMENTO"].fillna("").astype(str).str.strip(),
    )
    .query("municipio != '' and estabelecimento != ''")
    .groupby("municipio", as_index=False)
    .agg(hospitais=("estabelecimento", "nunique"))
    .sort_values(["hospitais", "municipio"], ascending=[False, True])
    .reset_index(drop=True)
)

tabela_regionais_hospitais = (
    base_hosp.assign(
        regional=base_hosp["regional_assoc"].fillna("Nenhum").astype(str).str.strip(),
        estabelecimento=base_hosp["ESTABELECIMENTO"].fillna("").astype(str).str.strip(),
    )
    .query("regional != '' and estabelecimento != ''")
    .groupby("regional", as_index=False)
    .agg(hospitais=("estabelecimento", "nunique"))
    .sort_values(["hospitais", "regional"], ascending=[False, True])
    .reset_index(drop=True)
)

print('Hospitais por município (estabelecimentos únicos):')
display(tabela_municipios_hospitais)

print('\nHospitais por regional (estabelecimentos únicos):')
display(tabela_regionais_hospitais)

municipios_com_hospital = combo_municipio["CODIGO_IBGE"].nunique()
regionais_com_hospital = base_hosp.loc[base_hosp["regional_assoc"] != "Nenhum", "regional_assoc"].nunique()

print(f"Municípios com pelo menos 1 hospital: {municipios_com_hospital}")
print(f"Regionais com pelo menos 1 hospital (sem repetir): {regionais_com_hospital}")
